This is one of the most important ideas in modern RAG systems.

Hybrid search is where retrieval becomes seriously powerful.

Let’s break it down in the same structured way:

👉 What

👉 Why

👉 Components (BM25, Semantic, RRF)

👉 Deep analogy

👉 How they work together

🧠 1. What is Hybrid Search?

🔍 Simple Definition

👉 Hybrid search = combining multiple retrieval methods to get better results

Keyword Search (BM25) + Semantic Search (Embeddings) → Combined Results

🔄 Pipeline View

User Query

   ↓

BM25 (Keyword Search)

   ↓

Semantic Search (Vector)

   ↓

Fusion (RRF)

   ↓

Final Ranked Documents

❗ 2. Why Hybrid Search Exists

Each retrieval method has different strengths and weaknesses.

🔴 Problem with BM25 (Keyword Search)

BM25 relies on exact words

Example:

Query: "heart attack causes"

But document says:

"myocardial infarction reasons"

👉 BM25 fails (no exact match)

🔵 Problem with Semantic Search

Semantic search understands meaning, but:

👉 Can miss exact keyword matches

Example:

Query: "Python list append syntax"

Semantic search may return:

"How to modify arrays in programming"

👉 Too generic

✅ Solution: Combine Both

👉 BM25 = exact match

👉 Semantic = meaning match

👉 Hybrid = best of both worlds

🔵 3. BM25 (Keyword Search)

🔍 What it does

👉 Matches based on:

exact words

frequency

importance

🧠 Analogy (Dictionary Search)

Imagine using a dictionary:

👉 You search:

"apple"

It gives entries with exact word "apple"

👉 That’s BM25

✅ Strength

Exact match

Works well for:

code

error messages

names

❌ Weakness

No understanding of meaning

🟢 4. Semantic Search (Vector Search)

🔍 What it does

👉 Converts text → embeddings → compares meaning

🧠 Analogy (Human Understanding)

You ask a friend:

"Why does sugar disease happen?"

Even if the term "diabetes" is not used, they understand.

👉 That’s semantic search

✅ Strength

Understands intent

Handles synonyms

Works for natural language

❌ Weakness

Can miss exact terms

Sometimes too “fuzzy”

🟣 5. RRF (Reciprocal Rank Fusion) ⭐

🔍 What it does

👉 Combines rankings from multiple retrievers

🧠 Key Idea

Instead of combining scores:

👉 Combine ranks

📊 Formula (Conceptual)

Score = 1 / (k + rank)

Where:

rank = position in list

k = smoothing constant (usually 60)

🧠 Analogy (Voting System)

Imagine 2 judges ranking contestants:

Judge 1 (BM25):

A → 1st

B → 2nd

C → 3rd

Judge 2 (Semantic):

B → 1st

C → 2nd

A → 3rd

👉 Who is best overall?

RRF combines rankings:

A → good in both

B → strong in one

C → moderate

👉 Final decision = balanced consensus

✅ Why RRF is Powerful

Doesn’t depend on score scale

Robust to noisy rankings

Works across different retrievers

🔥 6. Deep Analogy (Best Way to Understand Hybrid Search)

🎓 University Ranking System

You are selecting students using:

Method 1: Marks (BM25)

Exact scores

Objective

Method 2: Interview (Semantic)

Understanding

Subjective

Now:

👉 If you use only marks → miss talent

👉 If you use only interview → miss consistency

Solution: Combine both

👉 Use both rankings → final merit list

That’s:

Hybrid Search = Marks + Interview + Final Ranking (RRF)

🔄 7. How Hybrid Search Works Together

Step-by-step:

Step 1: BM25 Retrieval

Top 5 keyword-matching docs

Step 2: Semantic Retrieval

Top 5 meaning-based docs

Step 3: Combine using RRF

Merge rankings → final top docs

🧠 8. Example (End-to-End)

Query:

"How to reduce investment risk?"

BM25 Results:

1. "investment risk reduction strategies"

2. "financial risk management"

Semantic Results:

1. "diversification helps reduce losses"

2. "portfolio balancing techniques"

RRF Fusion:

Final Top Docs:

1. diversification helps reduce losses

2. investment risk reduction strategies

3. financial risk management

🧠 9. When to Use Hybrid Search

Scenario	             Use Hybrid?

Enterprise RAG	         ✅ MUST

Domain-specific systems	 ✅

Simple chatbot	         ❌ optional

Search engines	         ✅ critical

🔥 10. Common Mistakes

❌ Using only vector search

❌ Ignoring keyword matching

❌ Not using fusion (RRF)

❌ Combining scores incorrectly

🚀 11. Full Retrieval Stack (Modern RAG)

User Query

   ↓

Intent Validation
     
     |
     
Query Formulation

   ↓

Query Expansion

   ↓

Pre-filters
 
    |

Hybrid Search (BM25 + Semantic)

   ↓

RRF Fusion

   ↓

Post-filtering

   ↓

Reranking

   ↓

Final Context

   ↓

LLM Answer

🎯 Final Summary

Component	        Role

BM25	            Exact keyword match

Semantic	        Meaning-based match

RRF	                Combine both intelligently

Hybrid Search	    Best overall retrieval

🧠 Final Insight (Very Important)

👉 No single retrieval method is perfect.

BM25 → Precision (exact match)

Semantic → Recall (meaning match)

RRF → Balance (best ranking)

In [1]:
# ==========================================
# 1. SETUP
# ==========================================
import os
import json
import numpy as np
import chromadb
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY")

# OpenAI Chat Model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Embedding Model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


In [2]:
# ==========================================
# 2. DATA
# ==========================================
documents = [
    "Diabetes is a chronic disease affecting blood sugar levels.",
    "Hypertension increases risk of heart disease.",
    "Stock markets fluctuate due to economic conditions.",
    "Diversification reduces investment risk.",
    "Neural networks are key to deep learning.",
    "Cloud computing provides scalable infrastructure."
]

# BM25 setup
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

# Chroma setup
client = chromadb.Client()
collection = client.create_collection("hybrid_rag_full")

embeddings = [embedding_model.encode(doc).tolist() for doc in documents]
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=[str(i) for i in range(len(documents))]
)

In [3]:
# ==========================================
# 3. QUERY EXPANSION (LLM)
# ==========================================
def expand_query(query):
    prompt = f"""
    Generate 3 alternative search queries.
    Return JSON list.

    Query: {query}
    """
    response = llm.invoke(prompt)

    try:
        return json.loads(response.content)
    except:
        return [query]

In [4]:
# ==========================================
# 4. BM25 RETRIEVAL
# ==========================================
def bm25_retrieve(query, top_k=3):
    scores = bm25.get_scores(query.lower().split())
    ranked = np.argsort(scores)[::-1]
    return [documents[i] for i in ranked[:top_k]]

In [5]:
# ==========================================
# 5. VECTOR RETRIEVAL
# ==========================================
def vector_retrieve(query, top_k=3):
    q_emb = embedding_model.encode(query).tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    return results["documents"][0]

In [6]:
# ==========================================
# 6. RRF FUSION
# ==========================================
def rrf(rank_lists, k=60):
    scores = {}

    for rlist in rank_lists:
        for rank, doc in enumerate(rlist):
            scores[doc] = scores.get(doc, 0) + 1 / (k + rank)

    return sorted(scores, key=scores.get, reverse=True)

In [7]:
# ==========================================
# 7. HYBRID RETRIEVAL
# ==========================================
def hybrid_retrieval(query):

    # Step 1: Expand queries
    queries = expand_query(query)
    queries.append(query)

    all_rank_lists = []

    for q in queries:
        bm25_docs = bm25_retrieve(q)
        vec_docs = vector_retrieve(q)

        all_rank_lists.append(bm25_docs)
        all_rank_lists.append(vec_docs)

    # Step 2: Fuse rankings
    fused_docs = rrf(all_rank_lists)

    return fused_docs[:5]

In [8]:
# ==========================================
# 8. LLM RERANKING
# ==========================================
def rerank_llm(query, docs):
    scored_docs = []

    for doc in docs:
        prompt = f"""
        Score relevance from 0 to 1.

        Query: {query}
        Document: {doc}
        """

        try:
            score = float(llm.invoke(prompt).content.strip())
        except:
            score = 0.0

        scored_docs.append((doc, score))

    ranked = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked]

In [9]:
# ==========================================
# 9. FINAL GENERATION
# ==========================================
def generate_answer(query, docs):
    context = "\n".join(docs)

    prompt = f"""
    Answer ONLY using the context.

    Context:
    {context}

    Question: {query}
    """

    response = llm.invoke(prompt)
    return response.content

In [10]:
# ==========================================
# 10. FULL PIPELINE
# ==========================================
def hybrid_rag_pipeline(query):

    print("🔍 Query:", query)

    # Step 1: Hybrid Retrieval
    retrieved_docs = hybrid_retrieval(query)
    print("📄 Retrieved:", retrieved_docs)

    # Step 2: Rerank
    final_docs = rerank_llm(query, retrieved_docs)
    print("⭐ Reranked:", final_docs)

    # Step 3: Generate Answer
    answer = generate_answer(query, final_docs[:3])

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "final_docs": final_docs[:3],
        "answer": answer
    }

In [11]:
# ==========================================
# 11. TEST
# ==========================================
if __name__ == "__main__":

    queries = [
        "How to reduce investment risk?",
        "What is diabetes?",
        "Explain neural networks"
    ]

    for q in queries:
        print("\n========================")
        result = hybrid_rag_pipeline(q)
        print(json.dumps(result, indent=2))


🔍 Query: How to reduce investment risk?
📄 Retrieved: ['Diversification reduces investment risk.', 'Stock markets fluctuate due to economic conditions.', 'Neural networks are key to deep learning.', 'Hypertension increases risk of heart disease.']
⭐ Reranked: ['Diversification reduces investment risk.', 'Stock markets fluctuate due to economic conditions.', 'Neural networks are key to deep learning.', 'Hypertension increases risk of heart disease.']
{
  "query": "How to reduce investment risk?",
  "retrieved_docs": [
    "Diversification reduces investment risk.",
    "Stock markets fluctuate due to economic conditions.",
    "Neural networks are key to deep learning.",
    "Hypertension increases risk of heart disease."
  ],
  "final_docs": [
    "Diversification reduces investment risk.",
    "Stock markets fluctuate due to economic conditions.",
    "Neural networks are key to deep learning."
  ],
  "answer": "Diversification reduces investment risk."
}

🔍 Query: What is diabetes?
📄